# RealMLP — a neural-net alternative to CatBoost-V1

An approach adapted from [PSS6E7 RealMLP CV:0.95063](https://www.kaggle.com/code/yunsuxiaozi/pss6e7-realmlp-cv-0-95063)
by yunsuxiaozi -- our first neural-net model family in this project (everything
prior has been tree-boosting: LightGBM, CatBoost, XGBoost, HistGradientBoosting).

RealMLP is a from-scratch PyTorch architecture (periodic numeric embeddings,
NTK-parametrized linear layers, a 16-way ensemble-in-one-model trick, EMA of
weights) specifically tuned to be competitive with boosted trees on tabular
data. The source notebook's raw CV (before its own post-hoc correction) was
0.95057 -- essentially tied with our own v0.7 (0.9502).

Scope decisions vs. the source notebook:
- Our own `StratifiedKFold(5, shuffle=True, random_state=42)` instead of the
  source's naive `index % num_folds` fold assignment.
- Architecture kept faithful (PBLD embeddings, NTPLinear, categorical layer,
  RankGLU head, `n_ens=16`, EMA, 5-parameter-group LR schedule) -- the point is
  testing whether the architecture itself is competitive.
- The source notebook's own investigation (see
  `docs/investigate/2026-07-05-kaggle-discussion-findings.md`) found its
  post-hoc per-class Optuna reweighting adds only +0.00006 over the raw model --
  negligible, consistent with 5 prior independent findings that a tunable
  second correction on top of already-balanced training finds ~nothing extra.
  **Not reproduced here** -- single correction only (the architecture's own
  training-time class weighting), plain argmax.
- Reproduces CatBoost-V1 (our current second-best, v0.3's exact config) as a
  comparison peg and blend-check member, same pattern as every prior version.
- Run **locally first** (Apple Silicon, PyTorch MPS backend) before considering
  a Kaggle push.

In [1]:
import os
import glob
import math
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import TargetEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

DATA_DIR = Path('/kaggle/input/playground-series-s6e7')
if not (DATA_DIR / 'train.csv').exists():
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if hits:
        DATA_DIR = Path(hits[0]).parent
    else:
        DATA_DIR = Path('..') / 'data'  # local run
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
print('DATA_DIR:', DATA_DIR)

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'device: {device}')

def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(2026)

N_FOLDS = 5
EPOCHS = 5
TRAIN_BS = 512
EVAL_BS = 10240
EMBED_DIM = 4
LR = 0.01
N_ENS = 16
ONEHOTMAX = 10
LS = 0.05
EMA_DECAY = 0.997

TARGET = 'health_condition'
NUMERIC_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL_COLS = ['diet_type', 'stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol', 'gender']
BASE_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('train', train.shape, 'test', test.shape)

DATA_DIR: ../data
device: mps
train (690088, 15) test (295753, 14)


## Feature engineering

Matches the source notebook: numeric features get two extra binned copies at
different granularities (`_cat`, `_cat2`), all categorical/binned columns get
target-encoded per fold. This independently tests multi-resolution numeric
target encoding, a variant of our own v0.7's exact-value-only approach.

In [2]:
def FE(df):
    for c in NUMERIC_COLS:
        if c == 'step_count':
            df[f'{c}_cat'] = (df[c] // 10).astype(str)
        elif c == 'calorie_expenditure':
            df[f'{c}_cat'] = (df[c] // 5).astype(str)
        else:
            df[f'{c}_cat'] = df[c].astype(str)
    for c in NUMERIC_COLS:
        if c == 'step_count':
            df[f'{c}_cat2'] = (df[c] // 20).astype(str)
        elif c == 'calorie_expenditure':
            df[f'{c}_cat2'] = (df[c] // 50).astype(str)
        elif c == 'water_intake':
            df[f'{c}_cat2'] = (df[c] * 50).astype(np.int64).astype(str)
        elif c in ('heart_rate', 'bmi'):
            df[f'{c}_cat2'] = (df[c] * 5).astype(np.int64).astype(str)
        else:
            df[f'{c}_cat2'] = (df[c] // 2).astype(str)
    return df

for c in CATEGORICAL_COLS:
    train[c] = train[c].fillna('missing').astype(str)
    test[c] = test[c].fillna('missing').astype(str)
for c in NUMERIC_COLS:
    train[c] = train[c].fillna(0)
    test[c] = test[c].fillna(0)

train = FE(train)
test = FE(test)

CATS = [c for c in test.columns if not pd.api.types.is_numeric_dtype(test[c]) or train[c].nunique() <= ONEHOTMAX]
NUMS = [c for c in test.columns if c not in CATS and c != TARGET and c != 'id']
print(f'Categorical (incl. binned): {len(CATS)} -> {CATS}')
print(f'Numerical: {len(NUMS)} -> {NUMS}')

# integer-ID map categoricals (train-based vocabulary, unseen test values -> 0)
CAT_DIMS = []
for c in CATS:
    mapping = {v: i + 1 for i, v in enumerate(train[c].unique())}
    train[c] = train[c].map(mapping)
    test[c] = test[c].map(mapping).fillna(0).astype(np.int64)
    CAT_DIMS.append(int(train[c].max()) + 1)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[TARGET])
n_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))

Categorical (incl. binned): 20 -> ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'sleep_duration_cat', 'heart_rate_cat', 'bmi_cat', 'calorie_expenditure_cat', 'step_count_cat', 'exercise_duration_cat', 'water_intake_cat', 'sleep_duration_cat2', 'heart_rate_cat2', 'bmi_cat2', 'calorie_expenditure_cat2', 'step_count_cat2', 'exercise_duration_cat2', 'water_intake_cat2']
Numerical: 7 -> ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
classes: ['at-risk', 'fit', 'unhealthy']


## RealMLP architecture

Kept faithful to the source notebook: `PBLDEmbedding` (periodic numeric
embeddings), `CategoricalFeatureLayer` (binary/one-hot/embedding routing by
cardinality), `NTPLinear` (NTK-parametrized linear layers, weights normalized
by `1/sqrt(in_features)`), `ScalingLayer` (learnable per-feature scale), and a
`RankGLUHead` residual bottleneck-GLU classification head. `n_ens=16`
independent ensemble members are batched into one model/training run via an
extra tensor dimension.

In [3]:
class RobustScaleSmoothClipTransform(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self._median = np.median(X, axis=-2)
        quant_diff = np.quantile(X, 0.75, axis=-2) - np.quantile(X, 0.25, axis=-2)
        idxs = quant_diff == 0.0
        quant_diff[idxs] = 0.5 * (np.max(X, axis=-2)[idxs] - np.min(X, axis=-2)[idxs])
        factors = 1.0 / (quant_diff + 1e-30)
        factors[quant_diff == 0.0] = 0.0
        self._factors = factors
        return self

    def transform(self, X, y=None):
        x_scaled = self._factors[None, :] * (X - self._median[None, :])
        return x_scaled / np.sqrt(1 + (x_scaled / 3) ** 2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]


class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, device=None):
        super().__init__()
        self.n_ens = n_ens
        self.embed_dim = embed_dim
        self.cat_dims = cat_dims

        self.onehot_features, self.binary_features = [], []
        self.embed_features, self.embed_dims, self.embed_offsets = [], [], []

        for i, dim in enumerate(cat_dims):
            if dim == 2:
                self.binary_features.append(i)
            elif dim <= ONEHOTMAX:
                self.onehot_features.append(i)
            else:
                self.embed_features.append(i)
                self.embed_dims.append(dim)

        self.combined_emb = None
        if self.embed_features:
            total_vocab = int(sum(self.embed_dims) * n_ens)
            self.combined_emb = nn.Embedding(total_vocab, self.embed_dim, padding_idx=0)
            offset = 0
            for dim in self.embed_dims:
                self.embed_offsets.append(offset)
                offset += dim
            self.per_ens_offset = sum(self.embed_dims)

    def forward(self, x):
        batch_size, n_ens, n_cat = x.shape
        features = []

        if self.binary_features:
            binary_x = x[:, :, self.binary_features].float()
            features.append(2 * binary_x - 1)

        if self.onehot_features:
            onehot_x = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_onehot_dim = sum(onehot_dims)
            onehot_encoded = torch.zeros(batch_size, n_ens, total_onehot_dim, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx:idx + 1].long()
                pos = torch.clamp(pos, 0, dim - 1)
                onehot_encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(onehot_encoded)

        if self.embed_features and self.combined_emb is not None:
            embed_x = x[:, :, self.embed_features].long()
            ens_offset = torch.arange(n_ens, device=x.device) * self.per_ens_offset
            feat_offset = torch.tensor(self.embed_offsets, device=x.device)
            indices = embed_x + feat_offset.unsqueeze(0).unsqueeze(0) + ens_offset.unsqueeze(0).unsqueeze(-1)
            embedded = self.combined_emb(indices)
            features.append(embedded.view(batch_size, n_ens, -1))

        return torch.cat(features, dim=2)


class PBLDEmbedding(nn.Module):
    def __init__(self, n_ens, n_features, hidden_dim=16, out_dim=4, freq_scale=0.1):
        super().__init__()
        self.n_ens, self.n_features, self.out_dim = n_ens, n_features, out_dim
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) * (1.0 / np.sqrt(hidden_dim)))
        self.b2 = nn.Parameter(torch.randn(n_ens, n_features, out_dim - 1))
        self.act = nn.GELU()
        nn.init.uniform_(self.b1, -np.pi, np.pi)

    def forward(self, x):
        batch_size = x.shape[0]
        x_expanded = x.unsqueeze(-1)
        periodic = torch.cos(2 * np.pi * (x_expanded * self.w1.unsqueeze(0) + self.b1.unsqueeze(0)))
        transformed = torch.einsum('b n f h, n f h d -> b n f d', periodic, self.w2)
        transformed = self.act(transformed + self.b2.unsqueeze(0))
        result = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        return result.view(batch_size, self.n_ens, -1)


class NTPLinear(nn.Module):
    def __init__(self, n_ens, in_features, out_features, bias=True):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        x = torch.einsum('b n i, n i o -> b n o', x, self.weight) / np.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x


class RankGLUHead(nn.Module):
    def __init__(self, input_dim, bottleneck_dim=128, output_dim=1, gamma=0.5, n_ens=8):
        super().__init__()
        self.gamma = gamma
        self.layer_norm = nn.LayerNorm(input_dim)
        self.linear_score = nn.Linear(input_dim, output_dim)
        self.glu_v = nn.Linear(input_dim, bottleneck_dim)
        self.glu_g = nn.Linear(input_dim, bottleneck_dim)
        self.glu_out = nn.Linear(bottleneck_dim, output_dim)

    def forward(self, x):
        normalized = self.layer_norm(x)
        linear_part = self.linear_score(normalized)
        gate = torch.sigmoid(self.glu_g(normalized))
        value = self.glu_v(normalized)
        glu_output = self.glu_out(value * gate)
        return linear_part + self.gamma * glu_output


class RealMLP(nn.Module):
    def __init__(self, output_dim=3, cat_dims=(), n_numerical=None, n_ens=8, embed_dim=4):
        super().__init__()
        act = nn.GELU
        self.n_ens, self.embed_dim, self.output_dim = n_ens, embed_dim, output_dim

        self.cate = CategoricalFeatureLayer(n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim)
        self.num_embed = PBLDEmbedding(n_features=n_numerical, hidden_dim=20, out_dim=5,
                                        freq_scale=5.0, n_ens=n_ens)
        num_emb_dim = n_numerical * 5
        cat_emb_dim = sum(c if c <= ONEHOTMAX else embed_dim for c in cat_dims)
        total_dim = num_emb_dim + cat_emb_dim

        self.dropout = nn.Dropout(0.03)
        self.first_linear = NTPLinear(n_ens=n_ens, in_features=total_dim, out_features=256)
        self.model = nn.Sequential(
            ScalingLayer(n_ens=n_ens, n_features=total_dim), self.dropout,
            self.first_linear,
            NTPLinear(n_ens=n_ens, in_features=256, out_features=512), act(),
            NTPLinear(n_ens=n_ens, in_features=512, out_features=128), act(), self.dropout,
        )
        self.cls_head = RankGLUHead(input_dim=128, bottleneck_dim=128, output_dim=output_dim,
                                     gamma=0.5, n_ens=n_ens)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        x = self.model(torch.cat([x_num, x_cat], dim=2))
        return self.cls_head(x)


def get_parameter_groups_5class(model):
    first_linear_weight_id = id(model.first_linear.weight)
    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if 'scale' in name:
            scale_p.append(param)
        elif 'num_embed' in name:
            pbld_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif 'bias' in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)
    return scale_p, pbld_p, first_w_p, other_w_p, bias_p


def flat_anneal(init_value, progress, flat_ratio=0.5):
    if progress < flat_ratio:
        return init_value
    decay_progress = (progress - flat_ratio) / (1 - flat_ratio)
    return init_value * (1 - decay_progress)


def cos_schedule(init_value, progress):
    return init_value * (math.cos(math.pi * progress) + 1) / 2

## Training loop

Per fold: target-encode the categorical/binned columns (fit on the training
fold only), robust-scale the numerics, train RealMLP with the standard 5
parameter-group AdamW schedule and EMA, track best-epoch balanced accuracy.
Single correction only -- training-time `class_weight='balanced'` (with the
source's fixed `[0.9, 1.1, 1.0]` tweak) -- **no post-hoc reweighting**.

In [4]:
X_cat_all = train[CATS].values
oof_proba_realmlp = np.zeros((len(train), n_classes))
test_proba_realmlp = np.zeros((len(test), n_classes))

fold_bar = tqdm(folds, desc='RealMLP folds')
for fold, (tr_idx, val_idx) in enumerate(fold_bar):

    X_num_tr, X_cat_tr, y_tr = train[NUMS].values[tr_idx], X_cat_all[tr_idx], y[tr_idx]
    X_num_val, X_cat_val, y_val = train[NUMS].values[val_idx], X_cat_all[val_idx], y[val_idx]
    X_num_test, X_cat_test = test[NUMS].values.copy(), test[CATS].values.copy()

    te_cols = [c for c in CATS if 'cat' in c]
    encoder = TargetEncoder(cv=5, smooth='auto', target_type='multiclass')
    tr_enc = encoder.fit_transform(pd.DataFrame(X_cat_tr, columns=CATS)[te_cols], y_tr)
    val_enc = encoder.transform(pd.DataFrame(X_cat_val, columns=CATS)[te_cols])
    tst_enc = encoder.transform(pd.DataFrame(X_cat_test, columns=CATS)[te_cols])

    X_num_tr = np.concatenate([X_num_tr, tr_enc], axis=1)
    X_num_val = np.concatenate([X_num_val, val_enc], axis=1)
    X_num_test = np.concatenate([X_num_test, tst_enc], axis=1)

    unique_classes = np.unique(y_tr)
    fold_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_tr)
    fold_class_weights = torch.tensor(fold_class_weights * np.array([0.9, 1.1, 1.0]),
                                       dtype=torch.float32).to(device)
    fold_bar.set_postfix(class_weights=[round(w, 3) for w in fold_class_weights.tolist()])

    rssc = RobustScaleSmoothClipTransform()
    rssc.fit(X_num_tr)
    X_num_tr = rssc.transform(X_num_tr)
    X_num_val = rssc.transform(X_num_val)
    X_num_test = rssc.transform(X_num_test)

    realmlp = RealMLP(output_dim=n_classes, cat_dims=[d for d in CAT_DIMS],
                       n_numerical=X_num_tr.shape[1], n_ens=N_ENS, embed_dim=EMBED_DIM).to(device)

    scale_p, pbld_p, first_w_p, other_w_p, bias_p = get_parameter_groups_5class(realmlp)
    optimizer = torch.optim.AdamW([
        {'params': scale_p,   'lr': LR * 20.0,  'weight_decay': 1e-2 * 0.1},
        {'params': pbld_p,    'lr': LR * 0.093, 'weight_decay': 1e-2},
        {'params': first_w_p, 'lr': LR * 1.0,   'weight_decay': 1e-2 * 0.1},
        {'params': other_w_p, 'lr': LR,         'weight_decay': 1e-2},
        {'params': bias_p,    'lr': LR * 0.1,   'weight_decay': 1e-2 * 0.5},
    ], betas=(0.9, 0.98))

    X_num_tr_t = torch.as_tensor(X_num_tr, dtype=torch.float32, device=device)
    X_cat_tr_t = torch.as_tensor(X_cat_tr, dtype=torch.long, device=device)
    y_tr_t = torch.as_tensor(y_tr, dtype=torch.long, device=device)
    X_num_val_t = torch.as_tensor(X_num_val, dtype=torch.float32, device=device)
    X_cat_val_t = torch.as_tensor(X_cat_val, dtype=torch.long, device=device)
    X_num_test_t = torch.as_tensor(X_num_test, dtype=torch.float32, device=device)
    X_cat_test_t = torch.as_tensor(X_cat_test, dtype=torch.long, device=device)

    train_indices = np.arange(len(y_tr_t))
    ema_state = {k: v.detach().clone() for k, v in realmlp.state_dict().items()}

    best_acc, best_model_state = 0.0, None

    epoch_bar = tqdm(range(EPOCHS), desc=f'fold {fold} epochs', leave=False)
    for epoch in epoch_bar:
        batch_starts = range(0, len(y_tr_t), TRAIN_BS)
        epoch_loss_total, epoch_loss_n = 0.0, 0
        for idx in batch_starts:
            progress = epoch / EPOCHS + idx / (len(y_tr_t) * EPOCHS)
            bs_idx = train_indices[idx:idx + TRAIN_BS]
            realmlp.train()

            optimizer.param_groups[0]['lr'] = flat_anneal(LR * 20.0, progress)
            optimizer.param_groups[1]['lr'] = flat_anneal(LR * 0.093, progress)
            optimizer.param_groups[2]['lr'] = flat_anneal(LR * 1.0, progress)
            optimizer.param_groups[3]['lr'] = flat_anneal(LR, progress)
            optimizer.param_groups[4]['lr'] = flat_anneal(LR * 0.1, progress)

            optimizer.zero_grad()
            batch_x_num, batch_x_cat, batch_y = X_num_tr_t[bs_idx], X_cat_tr_t[bs_idx], y_tr_t[bs_idx]
            logits = realmlp(batch_x_num, batch_x_cat)
            loss = F.cross_entropy(
                logits.reshape(-1, n_classes), batch_y.repeat_interleave(N_ENS),
                weight=fold_class_weights, label_smoothing=cos_schedule(LS, progress),
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(realmlp.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss_total += loss.item()
            epoch_loss_n += 1

            with torch.no_grad():
                model_state = realmlp.state_dict()
                for key, value in model_state.items():
                    if torch.is_floating_point(value):
                        ema_state[key].mul_(EMA_DECAY).add_(value.detach(), alpha=1.0 - EMA_DECAY)
                    else:
                        ema_state[key].copy_(value)

        if epoch:
            realmlp.eval()
            live_state = {k: v.detach().clone() for k, v in realmlp.state_dict().items()}
            realmlp.load_state_dict(ema_state, strict=True)
            with torch.no_grad():
                all_probs = []
                for vi in range(0, len(y_val), EVAL_BS):
                    logits = realmlp(X_num_val_t[vi:vi + EVAL_BS], X_cat_val_t[vi:vi + EVAL_BS])
                    all_probs.append(F.softmax(logits, dim=-1).mean(dim=1).cpu().numpy())
                all_probs = np.concatenate(all_probs, axis=0)
                bal_acc = balanced_accuracy_score(y_val, all_probs.argmax(axis=1))
                if bal_acc > best_acc:
                    best_acc = bal_acc
                    best_model_state = {k: v.cpu().clone() for k, v in ema_state.items()}
            realmlp.load_state_dict(live_state, strict=True)

        epoch_bar.set_postfix(loss=f'{epoch_loss_total / max(epoch_loss_n, 1):.4f}', best_acc=f'{best_acc:.5f}')
        np.random.shuffle(train_indices)

    if best_model_state is not None:
        realmlp.load_state_dict(best_model_state)
    realmlp = realmlp.to(device)
    realmlp.eval()
    with torch.no_grad():
        all_probs = []
        for vi in range(0, len(y_val), EVAL_BS):
            logits = realmlp(X_num_val_t[vi:vi + EVAL_BS], X_cat_val_t[vi:vi + EVAL_BS])
            all_probs.append(F.softmax(logits, dim=-1).mean(dim=1).cpu().numpy())
        oof_proba_realmlp[val_idx] = np.concatenate(all_probs, axis=0)

        all_probs = []
        for ti in range(0, len(X_num_test_t), EVAL_BS):
            logits = realmlp(X_num_test_t[ti:ti + EVAL_BS], X_cat_test_t[ti:ti + EVAL_BS])
            all_probs.append(F.softmax(logits, dim=-1).mean(dim=1).cpu().numpy())
        test_proba_realmlp += np.concatenate(all_probs, axis=0) / N_FOLDS

oof_pred_realmlp = oof_proba_realmlp.argmax(axis=1)
oof_bal_acc_realmlp = balanced_accuracy_score(y, oof_pred_realmlp)
print(f'\nRealMLP OOF balanced accuracy: {oof_bal_acc_realmlp:.5f}')
print('source notebook raw CV (different fold split, before its own post-hoc correction): 0.95057')
print()
print(classification_report(y, oof_pred_realmlp, target_names=label_encoder.classes_, digits=4))

RealMLP folds:   0%|          | 0/5 [00:00<?, ?it/s]

fold 0 epochs:   0%|          | 0/5 [00:00<?, ?it/s]

fold 1 epochs:   0%|          | 0/5 [00:00<?, ?it/s]

fold 2 epochs:   0%|          | 0/5 [00:00<?, ?it/s]

fold 3 epochs:   0%|          | 0/5 [00:00<?, ?it/s]

fold 4 epochs:   0%|          | 0/5 [00:00<?, ?it/s]


RealMLP OOF balanced accuracy: 0.95062
source notebook raw CV (different fold split, before its own post-hoc correction): 0.95057

              precision    recall  f1-score   support

     at-risk     0.9938    0.9355    0.9637    592561
         fit     0.7283    0.9507    0.8248     39803
   unhealthy     0.6940    0.9657    0.8076     57724

    accuracy                         0.9389    690088
   macro avg     0.8054    0.9506    0.8654    690088
weighted avg     0.9534    0.9389    0.9427    690088



## CatBoost-V1 reproduction (v0.3's exact config, base features)

Same-data comparison peg (current second-best single model) and a blend check.

In [5]:
class BalancedAccuracyMetric:
    def is_max_optimal(self):
        return True
    def evaluate(self, approxes, target, weight):
        approx = np.array(approxes)
        pred = approx.argmax(axis=0)
        y_true = np.array(target).astype(int)
        return balanced_accuracy_score(y_true, pred), 1.0
    def get_final_error(self, error, weight):
        return error

train_cb = train[BASE_FEATURES].copy()
test_cb = test[BASE_FEATURES].copy()
for c in CATEGORICAL_COLS:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c] = test_cb[c].astype(str)

oof_proba_cb = np.zeros((len(train), n_classes))
test_proba_cb = np.zeros((len(test), n_classes))
cb_best_iters = []

for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc='CatBoost-V1 folds')):
    X_tr, X_val = train_cb.iloc[tr_idx], train_cb.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = CatBoostClassifier(
        loss_function='MultiClass', eval_metric=BalancedAccuracyMetric(),
        auto_class_weights='Balanced', iterations=3000, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, random_seed=42, task_type='CPU', verbose=False,
    )
    model.fit(X_tr, y_tr, cat_features=CATEGORICAL_COLS, eval_set=(X_val, y_val),
              early_stopping_rounds=100)
    oof_proba_cb[val_idx] = model.predict_proba(X_val)
    cb_best_iters.append(model.best_iteration_)
    test_proba_cb += model.predict_proba(test_cb) / N_FOLDS

oof_pred_cb = oof_proba_cb.argmax(axis=1)
oof_bal_acc_cb = balanced_accuracy_score(y, oof_pred_cb)
print('best_iterations:', cb_best_iters)
print(f'CatBoost-V1 OOF balanced accuracy: {oof_bal_acc_cb:.4f}')
print('v0.3 Variant 1 known result:       0.9493, best_iterations [428, 950, 605, 339, 779]')

CatBoost-V1 folds:   0%|          | 0/5 [00:00<?, ?it/s]

best_iterations: [765, 461, 703, 516, 576]
CatBoost-V1 OOF balanced accuracy: 0.9493
v0.3 Variant 1 known result:       0.9493, best_iterations [428, 950, 605, 339, 779]


In [6]:
assert abs(oof_bal_acc_cb - 0.9493) < 0.002, 'CatBoost-V1 reproduction did not match v0.3 Variant 1 closely enough'
print('CatBoost-V1 reproduction check: PASS')

CatBoost-V1 reproduction check: PASS


## Solo comparison

In [7]:
solo_scores = {'realmlp': oof_bal_acc_realmlp, 'catboost_v1': oof_bal_acc_cb}
for name, score in sorted(solo_scores.items(), key=lambda kv: -kv[1]):
    print(f'{score:.4f}  {name}')

0.9506  realmlp
0.9493  catboost_v1


## Blend weight search + nested validation

In [8]:
OOF_PROBAS = {'realmlp': oof_proba_realmlp, 'catboost_v1': oof_proba_cb}
TEST_PROBAS = {'realmlp': test_proba_realmlp, 'catboost_v1': test_proba_cb}
MEMBER_NAMES = ['realmlp', 'catboost_v1']

def pair_grid(step=0.02):
    n_steps = round(1 / step)
    return [(i * step, (n_steps - i) * step) for i in range(n_steps + 1)]

def blend_predict(probas_dict, weights, members):
    blended = sum(w * probas_dict[m] for w, m in zip(weights, members))
    return blended.argmax(axis=1)

def grid_search_blend(oof_probas_dict, y_true, members, grid):
    best_score, best_w = -1, tuple(1 / len(members) for _ in members)
    for w in tqdm(grid, desc=f'blend grid ({"+".join(members)})', leave=False):
        pred = blend_predict(oof_probas_dict, w, members)
        score = balanced_accuracy_score(y_true, pred)
        if score > best_score:
            best_score, best_w = score, w
    return best_score, best_w

grid2 = pair_grid(step=0.02)
best_score_2way, best_w_2way = grid_search_blend(OOF_PROBAS, y, MEMBER_NAMES, grid2)
print(f'2-way blend full-OOF grid search: best {best_score_2way:.4f} at weights {dict(zip(MEMBER_NAMES, (round(x,2) for x in best_w_2way)))}')
print(f'Best solo model:                  {max(solo_scores.values()):.4f}')
print(f'Improvement (same-data, likely optimistic): {best_score_2way - max(solo_scores.values()):+.4f}')

blend grid (realmlp+catboost_v1):   0%|          | 0/51 [00:00<?, ?it/s]

2-way blend full-OOF grid search: best 0.9507 at weights {'realmlp': 0.82, 'catboost_v1': 0.18}
Best solo model:                  0.9506
Improvement (same-data, likely optimistic): +0.0001


In [9]:
nested_scores_solo_best = []
nested_scores_blend = []
nested_weights_2way = []
best_solo_key = max(solo_scores, key=solo_scores.get)

for fold_idx, (_, val_idx) in enumerate(tqdm(folds, desc='nested CV (2-way blend)')):
    other_idx = np.concatenate([folds[j][1] for j in range(N_FOLDS) if j != fold_idx])
    other_oof = {m: OOF_PROBAS[m][other_idx] for m in MEMBER_NAMES}
    _, w_fold = grid_search_blend(other_oof, y[other_idx], MEMBER_NAMES, grid2)

    val_oof = {m: OOF_PROBAS[m][val_idx] for m in MEMBER_NAMES}
    blend_pred = blend_predict(val_oof, w_fold, MEMBER_NAMES)
    solo_pred = OOF_PROBAS[best_solo_key][val_idx].argmax(axis=1)

    nested_scores_solo_best.append(balanced_accuracy_score(y[val_idx], solo_pred))
    nested_scores_blend.append(balanced_accuracy_score(y[val_idx], blend_pred))
    nested_weights_2way.append(w_fold)

print('Per-fold blend weights (fit on the other 4 folds):', [tuple(round(x, 2) for x in w) for w in nested_weights_2way])
print(f'Nested best-solo-model ({best_solo_key}) balanced accuracy: {np.mean(nested_scores_solo_best):.4f} (+/- {np.std(nested_scores_solo_best):.4f})')
print(f'Nested 2-way-blend balanced accuracy:                      {np.mean(nested_scores_blend):.4f} (+/- {np.std(nested_scores_blend):.4f})')
print(f'Honest improvement estimate: {np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best):+.4f}')

nested CV (2-way blend):   0%|          | 0/5 [00:00<?, ?it/s]

blend grid (realmlp+catboost_v1):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (realmlp+catboost_v1):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (realmlp+catboost_v1):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (realmlp+catboost_v1):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (realmlp+catboost_v1):   0%|          | 0/51 [00:00<?, ?it/s]

Per-fold blend weights (fit on the other 4 folds): [(0.82, 0.18), (0.8, 0.2), (0.8, 0.2), (0.82, 0.18), (0.8, 0.2)]
Nested best-solo-model (realmlp) balanced accuracy: 0.9506 (+/- 0.0012)
Nested 2-way-blend balanced accuracy:                      0.9507 (+/- 0.0012)
Honest improvement estimate: +0.0001


## Decision + candidate submission

In [10]:
honest_improvement = np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best)
THRESHOLD_FOR_REAL_IMPROVEMENT = 0.0005
CURRENT_BEST_OOF = 0.9502  # v0.7 HGBC-TE, our current best model

print(f'RealMLP solo:              {oof_bal_acc_realmlp:.4f}')
print(f'CatBoost-V1 solo:          {oof_bal_acc_cb:.4f}')
print(f'Current project best OOF: {CURRENT_BEST_OOF:.4f}')
print()

if honest_improvement > THRESHOLD_FOR_REAL_IMPROVEMENT and (np.mean(nested_scores_blend) > CURRENT_BEST_OOF + THRESHOLD_FOR_REAL_IMPROVEMENT):
    print(f'REAL IMPROVEMENT: blend {honest_improvement:+.4f} over solo (nested-validated), and beats current best. Using weights fit on full OOF: {dict(zip(MEMBER_NAMES, best_w_2way))}')
    test_pred = blend_predict(TEST_PROBAS, best_w_2way, MEMBER_NAMES)
    submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    submission_path = OUTPUT_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows)')
elif oof_bal_acc_realmlp > CURRENT_BEST_OOF + THRESHOLD_FOR_REAL_IMPROVEMENT:
    print(f'RealMLP solo beats the current best by {oof_bal_acc_realmlp - CURRENT_BEST_OOF:+.4f} -- worth a submission on its own merits.')
    test_pred = test_proba_realmlp.argmax(axis=1)
    submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    submission_path = OUTPUT_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows)')
else:
    print(f'NO REAL IMPROVEMENT over the current best (~{CURRENT_BEST_OOF:.4f} OOF). '
          f'Not writing a new submission.csv.')

RealMLP solo:              0.9506
CatBoost-V1 solo:          0.9493
Current project best OOF: 0.9502

NO REAL IMPROVEMENT over the current best (~0.9502 OOF). Not writing a new submission.csv.


## Summary

**Flat result by our own strict threshold -- but the highest raw solo OOF of
any model in this project.** Run locally on Apple M3 Pro (PyTorch MPS
backend).

- **RealMLP solo OOF: 0.95062** -- edges out v0.7 (0.9502, our current best)
  by +0.0004, and closely matches the source notebook's own raw CV (0.95057).
  Per-class recall: at-risk 0.9355, fit 0.9507, unhealthy 0.9657.
- CatBoost-V1 reproduction: **PASS** (0.9493 OOF, matching v0.3 Variant 1
  exactly).
- Blend check: full-OOF grid search best 0.9507 at weights (realmlp=0.82,
  catboost_v1=0.18) -- not degenerate to one member, suggesting genuine
  diversity between a neural net and a GBDT. Nested-validated honest
  improvement over solo: only **+0.0001**.
- **Decision: NO REAL IMPROVEMENT.** RealMLP's raw margin over the current
  best (+0.0004) falls just short of the project's 0.0005 threshold
  (0.95062 vs. 0.9502 + 0.0005 = 0.9507) -- essentially a statistical tie
  with v0.7, not a confirmed new best. No submission written; nothing
  submitted to Kaggle.

**v0.7 (HGBC-TE) remains the best model in this project.** v0.8 is the
closest challenger yet, and the first genuinely different model family
(neural net, not another tree-boosting library) to reach parity -- worth
keeping as a candidate diversity source if blend work with v0.7 (rather
than CatBoost-V1) is revisited.